# SAM 2 for Roman Space Telescope: Tutorial 01
## Fast 'Segment Everything' & Decoupled Photometry on Astronomical FITS Images

Welcome to **sam2roman**! This tutorial demonstrates how to use Meta's **Segment Anything Model 2 (SAM 2)** via **Hugging Face** and **PyTorch** with high-performance single-pass vision caching, verbose diagnostics, and separate source photometry.

### Step 1: Import Libraries and Verify Environment

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import torch

import sam2roman
from sam2roman.io.fits import FITSImage
from sam2roman.inference.segmentor import SAM2Segmentor
from sam2roman.visualization.display import plot_masks_overlay, plot_detection_summary
from sam2roman.utils.synthetic import generate_synthetic_roman_fits

print(f"sam2roman version: {sam2roman.__version__}")
print(f"PyTorch version:   {torch.__version__}")
print(f"MPS (Apple GPU):   {torch.backends.mps.is_available()}")

### Step 2: Load Astronomical FITS Image

In [ ]:
fits_file = "sample_data/sample_roman_wfi.fits"
if not os.path.exists(fits_file):
    generate_synthetic_roman_fits(output_path=fits_file, shape=(384, 384), seed=2026)

astro_img = FITSImage.from_file(fits_file)
print(f"Image shape: {astro_img.shape}")
print(f"Filter:      {astro_img.header.get('FILTER', 'Unknown')}")

### Step 3: Initialize SAM 2 Segmentor

In [ ]:
segmentor = SAM2Segmentor(model_id="facebook/sam2.1-hiera-tiny")
print(f"Device: {segmentor.model_wrapper.device}")

### Step 4: Run High-Speed Segmentation ('Segment Everything')
Note: This step ONLY performs segmentation and mask extraction. Photometry is separated.

In [ ]:
results = segmentor.segment_everything(
    image=astro_img,
    prompt_mode="astronomical",  # or 'grid'
    pred_iou_thresh=0.65,
    min_mask_region_area=12,
    stretch_type="asinh",
    verbose=True,
)
print(f"Extracted {len(results)} sources!")

### Step 5: Separate Source Photometry
Now that segmentation is complete, we measure the aperture flux and SNR on the raw FITS data.

In [ ]:
import pandas as pd
catalog = results.compute_photometry(astro_img, verbose=True)
df = pd.DataFrame(catalog)
df.head(15)

### Step 6: Diagnostic Visualization

In [ ]:
fig = plot_detection_summary(
    image=astro_img,
    seg_result=results,
    title_prefix="Roman WFI (F158)",
    figsize=(16, 6),
)
plt.show()

### Step 7: Export to Multi-Extension FITS

In [ ]:
out_fits = "sample_data/sample_roman_wfi_segmented.fits"
astro_img.save_masks_to_fits(results.masks, output_path=out_fits)
print(f"Saved to {out_fits}")